# Informe tecnico - TP2 Vision por Computadora

Sistema de reconocimiento de razas de perros compuesto por tres etapas: busqueda por similitud, clasificacion supervisada y deteccion + clasificacion.

## Resumen ejecutivo

- Etapa 1: baseline `0.9286` y ResNet embeddings `0.9619` sobre 210 consultas.
- Etapa 2: ResNet18 fine-tuned `0.9429` accuracy test; CNN custom `0.2457`.
- Etapa 3: el pipeline final sobre una imagen de Beagle detecto al perro y clasifico la raza correctamente con score alto.

## Dataset

- Dataset: **70 Dog Breeds Image Dataset**.
- Splits usados: `train = 7946`, `valid = 700`, `test = 700`.
- Se detecto y corrigio una inconsistencia de nombres de clase entre splits (`American  Spaniel` vs `American Spaniel`).

## Implementacion por etapa

### Etapa 1
- Se implemento extraccion de embeddings baseline con ResNet18 preentrenado en ImageNet.
- Se agrego filtrado por metadata de modelo para evitar mezclar embeddings de `baseline`, `resnet18_finetuned` y `cnn_custom`.
- Se corrigio el indexado para permitir coexistencia de multiples modelos sobre la misma imagen.

### Etapa 2
- Se implemento entrenamiento/evaluacion de `resnet18_finetuned` y `cnn_custom`.
- Se guardan history, metricas, predicciones y metricas por clase en JSON.

### Etapa 3
- Se usa YOLOv8 preentrenado para detectar perros.
- Cada recorte se clasifica con el modelo entrenado en la Etapa 2.

## Resultados cuantitativos

### Clasificacion supervisada (test)

| Modelo | Accuracy | Precision | Recall | Specificity | F1 |
| --- | ---: | ---: | ---: | ---: | ---: |
| ResNet18 fine-tuned | 0.9429 | 0.9486 | 0.9429 | 0.9992 | 0.9432 |
| CNN custom | 0.2457 | 0.2908 | 0.2457 | 0.9891 | 0.2217 |

### Similitud (valid, 210 consultas)

| Modelo de embedding | Accuracy |
| --- | ---: |
| Baseline | 0.9286 |
| ResNet18 fine-tuned | 0.9619 |
| CNN custom | 0.3238 |

## Graficos

![Training curves](report_assets/classifier_history.png)

![Classifier metrics](report_assets/classifier_metrics.png)

![Similarity metrics](report_assets/similarity_metrics.png)

![ResNet confusion](report_assets/resnet_confusion_matrix.png)

![CNN confusion](report_assets/cnn_confusion_matrix.png)

## Analisis de errores

- El **ResNet18 fine-tuned** aprende rasgos discriminativos mucho mas robustos y conserva buena separacion entre razas visualmente cercanas.
- La **CNN custom** queda muy por debajo y concentra errores en clases de morfologia similar o con pocas senales distintivas.
- En deteccion + clasificacion, el error dominante deja de ser la deteccion y pasa a ser la calidad del clasificador usado sobre el recorte.

## Cambios adicionales justificados

- Se corrigio la carga de `.env` para que backend y scripts resuelvan siempre la configuracion esperada.
- Se mejoro `Dockerfile` para usar wheels CPU de PyTorch en Docker y reducir peso/fragilidad del build.
- Se arreglo la consulta a `pgvector` usando cast explicito a `vector`.
- Se agrego cache al store JSON y carga bulk en `build_index.py` para que el indexado completo sea practicable.

## Conclusiones

El sistema queda funcional en las tres etapas. La mejor configuracion final es usar **ResNet18 fine-tuned** como clasificador y tambien como extractor de embeddings para similitud. Con esa combinacion, el comportamiento observado es consistente tanto en metricas como en pruebas puntuales del pipeline completo.